In [1]:
import os
from tqdm.auto import tqdm
import pandas as pd

# to use data structures
from pydantic import BaseModel, ConfigDict
# from enum import Enum

# OpenAI API
from openai import OpenAI

# Load the API keys from .env
from dotenv import load_dotenv

pd.options.mode.chained_assignment = (
    None  # default='warn' This disables warning of "copying a slice of a DataFrame"
)
tqdm.pandas()  # activate the tqdm for pandas

In [2]:
# load the API keys
load_dotenv(".env")
for api_key in ["OPENAI_API_KEY"]:
    if os.getenv(api_key) is not None:
        print(api_key, "loaded")
    else:
        print(api_key, "missing")
        print("Please create a .env file with the corresponding API key")

OPENAI_API_KEY loaded


In [3]:
from utils.set_paths import (
    path_app_context,
    path_templates,
    path_data,
    path_output,
)

path_batch_files = path_data / "batches"

# import util functions
from utils.content_utils import *
from utils.file_ops import *

In [29]:
summary_excel = pd.read_excel("../EconJM_status.xlsx", sheet_name="sum")
summary_excel.tail(5)

,add_id,app_notes,deadline_target,add_posting,add_how_apply,good_match,recruitment_officer,app_status,outcome,network email,...,institution,location,add_rank,add_field,add_category,review_deadline,other_link,need_cover_letter,need_RS,need_TS
175,uta,NaN,2025-11-25,JOE,NaN,NaN,NaN,generating docs,NaN,NaN,...,UT Austin,NaN,assistant,NaN,research,2025-12-31,NaN,1,1,1
176,knoxville,NaN,2025-11-25,https://apply.interfolio.com/172749,NaN,NaN,NaN,generating docs,NaN,NaN,...,The University of Tennessee,NaN,assistant,NaN,research,2025-12-01,NaN,1,0,0
177,yale_cadmy,NaN,2025-12-01,JOE,NaN,NaN,NaN,generating docs,NaN,NaN,...,Yale,NaN,postdoc,NaN,research,2025-01-31,NaN,1,1,0
178,vermont,lecturer,2026-01-01,https://www.uvmjobs.com/,NaN,NaN,NaN,generating docs,NaN,NaN,...,University of Vermont,NaN,teaching,NaN,NaN,2026-01-19,NaN,1,0,1
179,columbia,NaN,NaT,EJM,https://apply.interfolio.com/175578,NaN,NaN,completed,NaN,NaN,...,Columbia University,NaN,assistant,NaN,research,NaT,NaN,0,0,0


# Load the needed input files

- Research Statement Template
- Cover Letter Template
- Teaching Statement Template

In [30]:
# Research Statement template
with open(path_templates / "base_text/research_statement.txt", "r") as f:
    lines = f.readlines()

RA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/teaching_statement.txt", "r") as f:
    lines = f.readlines()

TA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/cover_letter.txt", "r") as f:
    lines = f.readlines()

CL_TEMPLATE = " ".join(lines)

# Figure out what docs are needed to generate

In [31]:
columns_to_show = [
    "add_id",
    "institution",
    "location",
    "add_rank",
    "add_field",
    "add_category",
    "need_cover_letter",
    "need_RS",
    "need_TS",
]

In [32]:
completed_docs = [str(d).split("/")[-1] for d in path_output.iterdir() if d.is_dir()]

docs_to_generate = set(summary_excel.add_id.to_list()) - set(completed_docs)

if docs_to_generate:
    print("Need to generate docs for:")
    to_generate_df = summary_excel[summary_excel.add_id.isin(docs_to_generate)][
        columns_to_show
    ]
    docs_list = to_generate_df.astype(str).values.tolist()
    for id in docs_to_generate:
        print(f"- {id}")
else:
    print("No docs needed to generate")

Need to generate docs for:
- tufs
- ohio_state
- utax
- tulane
- berkeley
- vermont
- arkansas
- wooster
- montevideo
- u_chicago
- moscow
- albany
- columbia
- jilaee
- knoxville
- akron
- tennessee
- texas_state
- iowa
- toledo
- uta


In [33]:
to_generate_df.head()

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS
15,tennessee,The University of Tennessee at Chattanooga,NaN,assistant,NaN,research,1,1,1
46,albany,University at Albany,NaN,assistant,NaN,research,1,0,0
47,tulane,Tulane University,NaN,teaching,NaN,NaN,1,0,1
48,iowa,The University of Iowa,NaN,assistant,NaN,research,1,0,1
49,u_chicago,University of Chicago,NaN,assistant,NaN,research,1,1,1


# Add the context and additional data for each prompt

For these all the `raw` documents are transcribed in a few bullet points:

- For Job add use 
    ```
    summarize the job add in a few bullet points. Prioritize the attributes, qualitites they look for in a candidate
    ```
- For the isntitutional values use 
    ```
    summarize the institution mission and values in a few bullet points
    ```
- For econ research department use 
    ```
    summarize the departmen or institution or center values and research topics in a few bullet points. Make emphasis on potential co-authors and researcg/teaching groups described there and write it in the file
    ```

In [34]:
# get the add description context
to_generate_df.loc[:, "add_description"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START JOB DESCRIPTION",
        end_marker="END JOB DESCRIPTION",
    ),
    axis=1,
)

to_generate_df.loc[:, "econ_context"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START ECON DEPT",
        end_marker="END ECON DEPT",
    ),
    axis=1,
)

to_generate_df.loc[:, "institution_values"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START INSTITUTIONAL DESCRIPTION",
        end_marker="END INSTITUTIONAL DESCRIPTION",
    ),
    axis=1,
)

# OpenAI Batches

In [35]:
openai_client = OpenAI()
model_name = "gpt-4.1-mini"

In [36]:
# Load the OpenAI batching df
# load history of batches
if (path_batch_files / "batches_history.csv").exists():
    batch_history = pd.read_csv(path_batch_files / "batches_history.csv")
else:
    print("No history of batches found")
    batch_history = pd.DataFrame(
        columns=[
            "created_at",
            "description",
            "model",
            "cat_type",
            "batch_id",
            "status",
            "processing_status",
            "input_file_id",
            "output_file_id",
            "error_file_id",
        ]
    )

batch_history = update_batch_status(batch_history, openai_client)
batch_history.sort_values(["created_at", "model", "cat_type"], inplace=True)
batch_history.to_csv(path_batch_files / "batches_history.csv", index=False)


# Check the inprocess batches
in_process_batches = batch_history[batch_history["status"] == "in_progress"]
print("In-process batches:")
display(in_process_batches.tail(5))

# Check the completed batches
completed_batches = batch_history[
    (batch_history["status"] == "completed")
    & (batch_history["processing_status"] == False)
    & (batch_history["error_file_id"].isna())
]
print("Completed batches:")
display(completed_batches.tail(5))

In-process batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id


Completed batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id
63,2025-11-01 04:51:04.931747,Generate Cover Letters,gpt-4.1-mini,cl,batch_6905d80a8568819085295d51fd4a5d4f,completed,False,file-BfLbDJp69QprwJc5z22iob,file-DWAEF6FqwiZGh91yi7w4LK,NaN
64,2025-11-01 04:51:07.246537,Generate Research Statements,gpt-4.1-mini,rs,batch_6905d80c51348190b2ecbdd6ba335a73,completed,False,file-ReMpVrCAMQx9cq47dYieME,file-Ut7DoMUeLoVEwBEjt4oC6V,NaN
65,2025-11-01 04:51:09.843716,Generate Teaching Statements,gpt-4.1-mini,ts,batch_6905d80ef6908190999aa4bcecf648ae,completed,False,file-G3xtQkDJhm3j9VSes2f5HA,file-A3Gug82jwdHt9bpVYWA2Hx,NaN


In [37]:
# batch_history

In [38]:
class BodyText(BaseModel):
    body_text: str
    cot: str
    model_config = ConfigDict(extra="forbid")

# Cover Letter

In [39]:
gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()
# gen_aux

In [40]:
def cl_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to add a position fit paragraph to the following COVER LETTER START and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    Keep the length of the paragraph to about 100 words.
    Use a professional and academic tone, but make it engaging and easy to read.
    For each statement of fit add an EXAMPLE from my research, teaching, or service experience that demostrates the fit statement.
    For example, for interdisciplinary research you can add that I won Dedman College Interdisciplinary Institute’s Inaugural Graduate Student Summer Research and Writing Fellowship.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full new paragraph as `body_text` and a short chain-of-thought explanation of how you modified the BASE COVER LETTER START to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE COVER LETTER START.

    BASE COVER LETTER START:
    {CL_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [41]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: cl_prompt(row),
    axis=1,
)

In [42]:
name_jsonl = "gen_cl"
description_batch = "Generate Cover Letters"

content_schema = BodyText
cat_type = "cl"

In [43]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and CL
Updating the existing old_docs
Need to generate 10 CL docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_69062e78f6408190ac7691bf05cc532c and updated the batch history.


# Research Statement

In [44]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()
# gen_aux

In [45]:
def rs_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to add a final paragraph to the following BASE RESEARCH STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my research fit to the JOB ADVERTISEMENT qualities described.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the paragraph as `body_text` and a short chain-of-thought explanation of how you modified the BASE RESEARCH STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE RESEARCH STATEMENT:
    {RA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [46]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: rs_prompt(row),
    axis=1,
)

In [47]:
name_jsonl = "gen_rs"
description_batch = "Generate Research Statements"

content_schema = BodyText
cat_type = "rs"

In [48]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and RS
Updating the existing old_docs
Need to generate 6 RS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_69062e7ba6808190a109eb64aa14e21f and updated the batch history.


# Teaching Statement

In [49]:
gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()
# gen_aux

In [50]:
def ts_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE TEACHING STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my educational philosophy fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the teaching statement to about 300 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full teaching statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE TEACHING STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE TEACHING STATEMENT:
    {TA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [51]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: ts_prompt(row),
    axis=1,
)

In [52]:
name_jsonl = "gen_ts"
description_batch = "Generate Teaching Statements"

content_schema = BodyText
cat_type = "ts"

In [53]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and TS
Updating the existing old_docs
Need to generate 7 TS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_69062e7d2e188190a7fab2bd0387b77f and updated the batch history.


# Generate the Docs

Here we assume that all docs have been generated

In [78]:
def generate_docs(df, generated_df, gen_type="rs"):
    if gen_type == "rs":
        dir_name = "research_statement"
        file_name = "Gonzalez_RS.typ"
    elif gen_type == "ts":
        dir_name = "teaching_statement"
        file_name = "Gonzalez_TS.typ"
    elif gen_type == "cl":
        dir_name = "cover_letter"
        file_name = "Gonzalez_CL.typ"
    else:
        raise ValueError("gen_type not recognized")

    for i, row in df.iterrows():
        print(f"Processing {row.add_id}...")

        new_content = generated_df[
            generated_df.add_id == row.add_id
        ].body_text_1.values[0]

        target_path = path_output / row.add_id
        subdir_path = target_path / dir_name

        if not target_path.exists():
            print(f"Creating directory: {target_path}")
            copy_and_rename_dir(path_templates / f"{dir_name}", target_path, dir_name)
        else:
            print(f"Main Directory already exists: {target_path}")
            if not subdir_path.exists():
                print(f"Creating subdirectory: {subdir_path}")
                copy_and_rename_dir(
                    path_templates / f"{dir_name}", target_path, dir_name
                )

        try:
            add_lines_to_typs_doc(subdir_path / file_name, new_content)
            print(f"{file_name} added for {row.add_id}")
        except Exception as e:
            print(f"Error processing {row.add_id}: {e}")

## Cover Letter

In [79]:
cl_gen_df = pd.read_csv(path_output / "cl_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()

generate_docs(gen_aux, cl_gen_df, gen_type="cl")

Processing utah...
Creating directory: ../output_docs/utah
Gonzalez_CL.typ added for utah
Processing uw_1...
Creating directory: ../output_docs/uw_1
Gonzalez_CL.typ added for uw_1
Processing oklahoma...
Creating directory: ../output_docs/oklahoma
Gonzalez_CL.typ added for oklahoma
Processing south_main...
Creating directory: ../output_docs/south_main
Gonzalez_CL.typ added for south_main
Processing purdue...
Creating directory: ../output_docs/purdue
Gonzalez_CL.typ added for purdue
Processing n_iowa...
Creating directory: ../output_docs/n_iowa
Gonzalez_CL.typ added for n_iowa
Processing pittsburg...
Creating directory: ../output_docs/pittsburg
Gonzalez_CL.typ added for pittsburg
Processing notredame...
Creating directory: ../output_docs/notredame
Gonzalez_CL.typ added for notredame
Processing uw_3...
Creating directory: ../output_docs/uw_3
Gonzalez_CL.typ added for uw_3
Processing south_carolina...
Creating directory: ../output_docs/south_carolina
Gonzalez_CL.typ added for south_carolin

## Research Statement

In [80]:
rs_gen_df = pd.read_csv(path_output / "rs_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()

generate_docs(gen_aux, rs_gen_df, gen_type="rs")

Processing utah...
Main Directory already exists: ../output_docs/utah
Creating subdirectory: ../output_docs/utah/research_statement
Gonzalez_RS.typ added for utah
Processing uw_1...
Main Directory already exists: ../output_docs/uw_1
Creating subdirectory: ../output_docs/uw_1/research_statement
Gonzalez_RS.typ added for uw_1
Processing south_main...
Main Directory already exists: ../output_docs/south_main
Creating subdirectory: ../output_docs/south_main/research_statement
Gonzalez_RS.typ added for south_main
Processing purdue...
Main Directory already exists: ../output_docs/purdue
Creating subdirectory: ../output_docs/purdue/research_statement
Gonzalez_RS.typ added for purdue
Processing n_iowa...
Main Directory already exists: ../output_docs/n_iowa
Creating subdirectory: ../output_docs/n_iowa/research_statement
Gonzalez_RS.typ added for n_iowa
Processing pittsburg...
Main Directory already exists: ../output_docs/pittsburg
Creating subdirectory: ../output_docs/pittsburg/research_statemen

## Teaching Statement

In [81]:
rt_gen_df = pd.read_csv(path_output / "ts_docs_gpt-4.1-mini.csv")

gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()

generate_docs(gen_aux, rt_gen_df, gen_type="ts")

Processing utah...
Main Directory already exists: ../output_docs/utah
Creating subdirectory: ../output_docs/utah/teaching_statement
Gonzalez_TS.typ added for utah
Processing uw_1...
Main Directory already exists: ../output_docs/uw_1
Creating subdirectory: ../output_docs/uw_1/teaching_statement
Gonzalez_TS.typ added for uw_1
Processing south_main...
Main Directory already exists: ../output_docs/south_main
Creating subdirectory: ../output_docs/south_main/teaching_statement
Gonzalez_TS.typ added for south_main
Processing purdue...
Main Directory already exists: ../output_docs/purdue
Creating subdirectory: ../output_docs/purdue/teaching_statement
Gonzalez_TS.typ added for purdue
Processing n_iowa...
Main Directory already exists: ../output_docs/n_iowa
Creating subdirectory: ../output_docs/n_iowa/teaching_statement
Gonzalez_TS.typ added for n_iowa
Processing pittsburg...
Main Directory already exists: ../output_docs/pittsburg
Creating subdirectory: ../output_docs/pittsburg/teaching_statemen